In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import lognorm, genextreme
from scipy.optimize import minimize

lower = [1.0, 0.5, 0.0, 50.0, 200.0, temp_lo[1], -1.0, 0.0]
upper = [5.0, 1.5, 2.0, 200.0, 1000.0, temp_hi[1], 1.0, 10.0]
p0 = [3.0, 1.0, 1.0, 100.0, 800.0, temp_obs[1], 0.0, 5.0]

result = minimize(lambda params: -ar_loglik(params, temp_obs, temp_sd, ebm_wrap), p0, bounds=list(zip(lower, upper)))
θ_ar = result.x

n_boot = 1000
boot_samp = np.random.gumbel(θ_gev[1], θ_gev[2], (dat_annmax.shape[0], n_boot))
rp_boot = np.quantile(boot_samp, 0.99, axis=0)

pfit = plt.figure()
x = np.linspace(1, 1.75, 100)
plt.plot(x, genextreme.pdf(x, θ_gev[1], loc=θ_gev[2], scale=θ_gev[3]), color='darkorange', linewidth=3, label="GEV Model")
plt.axvline(rp_emp, color='black', linewidth=3, linestyle='--', label="Empirical Return Level")
plt.xlim(1, 1.75)
plt.scatter(rp_boot[0], np.zeros(dat_annmax.shape[0]), color='orange', label="GEV Bootstrap Replicates", s=3, alpha=0.3)
plt.axvline(np.mean(rp_boot), color='orange', linewidth=3, label="GEV Bootstrap Estimate")
plt.axvline(2 * rp_emp - np.mean(rp_boot), color='orange', linewidth=3, linestyle=':', label="Bias-Corrected Estimate (GEV)")
plt.xlabel("Annual Maximum Storm Tide (m)")
plt.ylabel("Probability Density")
plt.legend()

phist = plt.figure()
plt.hist(rp_boot, bins=30, alpha=0.4, color='darkorange', label="GEV Bootstrap Samples")
plt.axvline(rp_emp, color='black', linewidth=3, linestyle='--', label="Empirical Estimate")
q_boot = 2 * rp_emp - np.quantile(rp_boot, [0.975, 0.025])
plt.axvline(np.mean(rp_boot), color='red', linestyle='--', linewidth=3, label="GEV Bootstrap Estimate")
plt.axvline(2 * rp_emp - np.mean(rp_boot), color='red', linestyle=':', linewidth=3, label="Bias-Corrected GEV Estimate")
plt.fill_betweenx([0, plt.ylim()[1]], q_boot[0], q_boot[1], color='orange', alpha=0.3, label="95% GEV Bootstrap CI")
plt.xlabel("100-Year Return Period Estimate (m)")
plt.ylabel("Count")
plt.legend()

# re-do bootstrap with lognormal distribution
init_θ = [1.0, 1.0]
lb = [0.0, 0.0]
ub = [5.0, 10.0]

def loglik_ln(θ):
    return -np.sum(lognorm.pdf(dat_annmax.residual, s=θ[1], scale=np.exp(θ[0])))

result_ln = minimize(loglik_ln, init_θ, bounds=list(zip(lb, ub)))
θ_ln = result_ln.x

boot_samp_ln = np.random.lognormal(θ_ln[0], θ_ln[1], (dat_annmax.shape[0], n_boot))
rp_boot_ln = np.quantile(boot_samp_ln, 0.99, axis=0)
q_boot_ln = 2 * rp_emp - np.quantile(rp_boot_ln, [0.975, 0.025])

# plot confidence intervals and estimates
# GEV fit
plt.figure(pfit.number)
plt.plot(x, genextreme.pdf(x, θ_gev[1], loc=θ_gev[2], scale=θ_gev[3]), color='darkorange', linewidth=3, label="GEV Model")
plt.axvline(rp_emp, color='black', linewidth=3, linestyle='--', label="Empirical Return Level")
plt.axvline(np.mean(rp_boot), color='orange', linewidth=3, label=False)
plt.axvline(2 * rp_emp - np.mean(rp_boot), color='orange', linewidth=3, linestyle=':', label=False)
# lognormal fit
plt.plot(x, lognorm.pdf(x, s=θ_ln[1], scale=np.exp(θ_ln[0])), color='darkgreen', lw=3, label="LogNormal Model")
plt.axvline(np.mean(rp_boot_ln), color='green', linewidth=3, label=False)
plt.axvline(2 * rp_emp - np.mean(rp_boot_ln), color='green', linewidth=3, linestyle=':', label=False)
plt.gcf().set_size_inches(5.5, 5.5)

# GEV histogram
plt.figure(phist.number)
plt.hist(rp_boot, bins=30, alpha=0.4, color='darkorange', label="GEV Bootstrap Samples")
plt.axvline(2 * rp_emp - np.mean(rp_boot), color='red', linestyle=':', linewidth=3)
plt.fill_betweenx([0, plt.ylim()[1]], q_boot[0], q_boot[1], color='orange', alpha=0.3)
plt.hist(rp_boot_ln, bins=30, alpha=0.3, color='green', label="LN Bootstrap Samples")
plt.axvline(2 * rp_emp - np.mean(rp_boot_ln), color='purple', linestyle=':', linewidth=3)
plt.fill_betweenx([0, plt.ylim()[1]], q_boot_ln[0], q_boot_ln[1], color='darkgreen', alpha=0.3)
plt.gcf().set_size_inches(5.5, 5.5)

plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm, genextreme
from scipy.optimize import minimize

n = 10000

# get model hindcasts
temp_iid = ebm_wrap(θ_iid[:-1])
temp_ar = ebm_wrap(θ_ar[:-2])

# get iid and AR residuals from relevant processes
residuals_iid = np.stack([np.random.normal(0, np.sqrt(temp_sd[i]**2 + θ_iid[-1]**2), n) for i in range(len(temp_sd))], axis=0)
residuals_ar = np.zeros((len(hind_idx), n))
for t in range(len(temp_sd)):
    if t == 0:
        residuals_ar[t, :] = np.random.normal(0, np.sqrt(θ_ar[-1]**2 / (1 - θ_ar[-2]**2) + temp_sd[0]**2), n)
    else:
        residuals_ar[t, :] = θ_ar[-2] * residuals_ar[t-1, :] + np.random.normal(0, np.sqrt(θ_ar[-1]**2 + temp_sd[t]**2), n)

# add residuals back to model simulations
model_sim_iid = (residuals_iid + temp_iid).T
model_sim_ar = (residuals_ar + temp_ar).T

# get quantiles
q90_iid = np.quantile(model_sim_iid, [0.05, 0.5, 0.95], axis=0)  # compute 90% prediction interval
q90_ar = np.quantile(model_sim_ar, [0.05, 0.5, 0.95], axis=0)  # compute 90% prediction interval

plt.errorbar(time_obs, temp_obs, yerr=(temp_obs - temp_lo, temp_hi - temp_obs), color='black', label="Observations", fmt='o', markersize=5)
plt.fill_between(hind_years, q90_iid[0, :], q90_iid[2, :], color='orange', alpha=0.2, label="IID")
plt.fill_between(hind_years, q90_ar[0, :], q90_ar[2, :], color='blue', alpha=0.2, label="AR")
plt.ylabel("(°C)")
plt.xlabel("Year")
plt.title("Temperature Anomaly")
plt.legend()
plt.show()



In [ ]:
# function to fit GEV model for each data set
init_θ = [1.0, 1.0, 0.0]
lb = [0.0, 0.0, -2.0]
ub = [5.0, 10.0, 2.0]

def loglik_gev(θ):
    return -np.sum(genextreme.logpdf(dat_annmax.residual, θ[0], loc=θ[1], scale=θ[2]))

# get estimates from observations
rp_emp = np.quantile(dat_annmax.residual, 0.99)
result = minimize(loglik_gev, init_θ, bounds=list(zip(lb, ub)))
θ_gev = result.x

plt.hist(dat_annmax.residual, density=True, bins=30, alpha=0.5, label="Annual Maximum Storm Tide (m)")
x = np.linspace(1, 2, 100)
plt.plot(x, genextreme.pdf(x, θ_gev[0], loc=θ_gev[1], scale=θ_gev[2]), linewidth=3, color='orange', label="Parametric Model")
plt.axvline(rp_emp, color='red', linewidth=3, linestyle='--', label="Empirical Return Level")
plt.axvline(np.quantile(genextreme.rvs(θ_gev[0], loc=θ_gev[1], scale=θ_gev[2], size=10000), 0.99), color='blue', linewidth=3, linestyle='--', label="Model Return Level")
plt.xlim(1, 2)
plt.legend()
plt.show()

n_boot = 1000
boot_samp = genextreme.rvs(θ_gev[0], loc=θ_gev[1], scale=θ_gev[2], size=(len(dat_annmax), n_boot))
rp_boot = np.quantile(boot_samp, 0.99, axis=0)

plt.plot(x, genextreme.pdf(x, θ_gev[0], loc=θ_gev[1], scale=θ_gev[2]), color='darkorange', linewidth=3, label="GEV Model")
plt.axvline(rp_emp, color='black', linewidth=3, linestyle='--', label="Empirical Return Level")
plt.xlim(1, 1.75)
plt.scatter(rp_boot, np.zeros(len(dat_annmax)), color='orange', label="GEV Bootstrap Replicates", s=3, alpha=0.3)
plt.axvline(np.mean(rp_boot), color='orange', linewidth=3, label="GEV Bootstrap Estimate")
plt.axvline(2 * rp_emp - np.mean(rp_boot), color='orange', linewidth=3, linestyle=':', label="Bias-Corrected Estimate (GEV)")
plt.legend()
plt.show()

plt.hist(rp_boot, bins=30, alpha=0.4, color='darkorange', label="GEV Bootstrap Samples")
plt.axvline(rp_emp, color='black', linewidth=3, linestyle='--', label="Empirical Estimate")
q_boot = 2 * rp_emp - np.quantile(rp_boot, [0.975, 0.025])
plt.axvline(np.mean(rp_boot), color='red', linestyle='--', linewidth=3, label="GEV Bootstrap Estimate")
plt.axvline(2 * rp_emp - np.mean(rp_boot), color='red', linestyle=':', linewidth=3, label="Bias-Corrected GEV Estimate")
plt.fill_betweenx([0, plt.ylim()[1]], q_boot[0], q_boot[1], color='orange', alpha=0.3, label="95% GEV Bootstrap CI")
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

k = 20
n_blocks = len(surge_resids) - k + 1
blocks = np.zeros((k, n_blocks))
for i in range(n_blocks):
    blocks[:, i] = surge_resids[i:(k + i)]

blocks[:, :5]

m = int(np.ceil(len(surge_resids) / k))
n_boot = 1000
surge_bootstrap = np.zeros((len(surge_resids), n_boot))
for i in range(n_boot):
    block_sample_idx = np.random.choice(range(n_blocks), m, replace=True)
    surge_bootstrap[:, i] = np.concatenate(blocks[:, block_sample_idx])

plt.figure()
plt.plot(surge_resids, color='black', linewidth=3, label="Data")
plt.xlabel("Hour")
plt.ylabel("(m)")
plt.title("Tide Gauge Residuals")
plt.alpha(0.5)
plt.plot(surge_bootstrap[:, 0], color='blue', linewidth=3, label="Replicate", alpha=0.5)
plt.show()

plt.figure()
plt.xlabel("Hour")
plt.ylabel("(m)")
plt.title("Tide Gauge Residuals")
for i in range(10):
    label = "Replicate" if i == 0 else False
    plt.plot(surge_bootstrap[:, i], label=label, color='gray', alpha=0.2, linewidth=2)
plt.plot(surge_resids, label="Data", color='black', linewidth=3)
plt.show()